In [ ]:
import pandas as pd
df = pd.read_csv("/content/102_Nutrients_texual.csv")

In [ ]:
df["text"] = (
    df["Main_food_description"].fillna("")
    + " "
    + df["catname"].fillna("")
    + " "
    + df["macroclass"].fillna("")
)

In [ ]:
# Generate embeddings

MODEL_NAME = "gpt2"

from transformers import AutoTokenizer
from transformers import AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_NAME
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

print(device)

cuda


In [ ]:
# Embedding extraction

import torch
import numpy as np
from tqdm import tqdm

def generate_embeddings_batch(
    texts,
    batch_size=32
):

    all_embeddings = []

    for i in tqdm(
        range(0, len(texts), batch_size)
    ):

        batch_texts = texts[
            i : i + batch_size
        ]

        tokens = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        tokens = {
            k: v.to(device)
            for k, v in tokens.items()
        }

        with torch.no_grad():

            outputs = model(**tokens)

            embeddings = (
                outputs.last_hidden_state[:, -1, :]
                .cpu()
                .numpy()
            )

        all_embeddings.append(
            embeddings
        )

    return np.vstack(
        all_embeddings
    )

In [ ]:
# Create embedding matrix

embeddings = generate_embeddings_batch(
    df["text"].tolist(),
    batch_size=32
)

100%|██████████| 93/93 [00:05<00:00, 16.61it/s]


In [ ]:
np.save(
    "gpt2_embeddings.npy",
    embeddings
)

print(
    "saved",
    embeddings.shape
)

saved (2970, 768)


In [ ]:
# Create nutrient matrix

meta_cols = [
"Food_code",
"Main_food_description",
"catnumb",
"catname",
"novaclass",
"macroclass",
"text"]

nutrient_cols = [
    col
    for col in df.columns
    if col not in meta_cols
]

X_nutrients = df[nutrient_cols].values

In [ ]:
X_nutrients.shape # (2970,102)

(2970, 103)

In [ ]:
# target

y = df["novaclass"]

In [ ]:
# cross-validation
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np
import pandas as pd

# Encode target labels

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Models

models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

# Train and Evaluate

results = []

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    acc_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    mcc_scores = []

    for train_idx, test_idx in skf.split(
        X_nutrients,
        y_encoded
    ):

        # Split labels

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        # Nutrient Features

        X_train_nutrient = X_nutrients[train_idx, :]
        X_test_nutrient = X_nutrients[test_idx, :]

        scaler = StandardScaler()

        X_train_nutrient = scaler.fit_transform(
            X_train_nutrient
        )

        X_test_nutrient = scaler.transform(
            X_test_nutrient
        )

        # gpt2 Embeddings

        X_train_embed = embeddings[train_idx]
        X_test_embed = embeddings[test_idx]

        # Combine Features

        X_train = np.hstack(
            (
                X_train_nutrient,
                X_train_embed
            )
        )

        X_test = np.hstack(
            (
                X_test_nutrient,
                X_test_embed
            )
        )

        # SMOTE

        smote = SMOTE(
            random_state=42
        )

        X_train_resampled, y_train_resampled = (
            smote.fit_resample(
                X_train,
                y_train
            )
        )

        # Train

        model.fit(
            X_train_resampled,
            y_train_resampled
        )

        # Predict

        y_pred = model.predict(
            X_test
        )

        # Metrics

        acc_scores.append(
            accuracy_score(
                y_test,
                y_pred
            )
        )

        precision_scores.append(
            precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        f1_scores.append(
            f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        mcc_scores.append(
            matthews_corrcoef(
                y_test,
                y_pred
            )
        )

    results.append({

        "Model": model_name,

        "Accuracy":
            np.mean(acc_scores),

        "Precision":
            np.mean(precision_scores),

        "Recall":
            np.mean(recall_scores),

        "F1":
            np.mean(f1_scores),

        "MCC":
            np.mean(mcc_scores)
    })

# Results Table

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

print("\nFINAL RESULTS\n")

display(results_df)


Training Random Forest...

FINAL RESULTS



,Model,Accuracy,Precision,Recall,F1,MCC
0,Random Forest,0.926263,0.926616,0.926263,0.925436,0.836992


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np
import pandas as pd

# Encode target labels

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Models

models = {

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        objective="multi:softmax",
        eval_metric="mlogloss",
        random_state=42
    )
}

# Train and Evaluate

results = []

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    acc_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    mcc_scores = []

    for train_idx, test_idx in skf.split(
        X_nutrients,
        y_encoded
    ):

        # Split labels

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        # Nutrient Features

        X_train_nutrient = X_nutrients[train_idx, :]
        X_test_nutrient = X_nutrients[test_idx, :]

        scaler = StandardScaler()

        X_train_nutrient = scaler.fit_transform(
            X_train_nutrient
        )

        X_test_nutrient = scaler.transform(
            X_test_nutrient
        )

        # gpt2 Embeddings

        X_train_embed = embeddings[train_idx]
        X_test_embed = embeddings[test_idx]

        # Combine Features

        X_train = np.hstack(
            (
                X_train_nutrient,
                X_train_embed
            )
        )

        X_test = np.hstack(
            (
                X_test_nutrient,
                X_test_embed
            )
        )

        # SMOTE

        smote = SMOTE(
            random_state=42
        )

        X_train_resampled, y_train_resampled = (
            smote.fit_resample(
                X_train,
                y_train
            )
        )

        # Train

        model.fit(
            X_train_resampled,
            y_train_resampled
        )

        # Predict

        y_pred = model.predict(
            X_test
        )

        # Metrics

        acc_scores.append(
            accuracy_score(
                y_test,
                y_pred
            )
        )

        precision_scores.append(
            precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        f1_scores.append(
            f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        mcc_scores.append(
            matthews_corrcoef(
                y_test,
                y_pred
            )
        )

    results.append({

        "Model": model_name,

        "Accuracy":
            np.mean(acc_scores),

        "Precision":
            np.mean(precision_scores),

        "Recall":
            np.mean(recall_scores),

        "F1":
            np.mean(f1_scores),

        "MCC":
            np.mean(mcc_scores)
    })

# Results Table

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

print("\nFINAL RESULTS\n")

display(results_df)


Training XGBoost...

FINAL RESULTS



,Model,Accuracy,Precision,Recall,F1,MCC
0,XGBoost,0.932323,0.932932,0.932323,0.932249,0.85165


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np
import pandas as pd

# Encode target labels

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Models

models = {

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1
    )
}

# Train and Evaluate

results = []

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    acc_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    mcc_scores = []

    for train_idx, test_idx in skf.split(
        X_nutrients,
        y_encoded
    ):

        # Split labels

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        # Nutrient Features

        X_train_nutrient = X_nutrients[train_idx, :]
        X_test_nutrient = X_nutrients[test_idx, :]

        scaler = StandardScaler()

        X_train_nutrient = scaler.fit_transform(
            X_train_nutrient
        )

        X_test_nutrient = scaler.transform(
            X_test_nutrient
        )

        # gpt2 Embeddings

        X_train_embed = embeddings[train_idx]
        X_test_embed = embeddings[test_idx]

        # Combine Features

        X_train = np.hstack(
            (
                X_train_nutrient,
                X_train_embed
            )
        )

        X_test = np.hstack(
            (
                X_test_nutrient,
                X_test_embed
            )
        )

        # SMOTE

        smote = SMOTE(
            random_state=42
        )

        X_train_resampled, y_train_resampled = (
            smote.fit_resample(
                X_train,
                y_train
            )
        )

        # Train

        model.fit(
            X_train_resampled,
            y_train_resampled
        )

        # Predict

        y_pred = model.predict(
            X_test
        )

        # Metrics

        acc_scores.append(
            accuracy_score(
                y_test,
                y_pred
            )
        )

        precision_scores.append(
            precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        f1_scores.append(
            f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        mcc_scores.append(
            matthews_corrcoef(
                y_test,
                y_pred
            )
        )

    results.append({

        "Model": model_name,

        "Accuracy":
            np.mean(acc_scores),

        "Precision":
            np.mean(precision_scores),

        "Recall":
            np.mean(recall_scores),

        "F1":
            np.mean(f1_scores),

        "MCC":
            np.mean(mcc_scores)
    })

# Results Table

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

print("\nFINAL RESULTS\n")

display(results_df)


Training LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



FINAL RESULTS



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1,MCC
0,LightGBM,0.934343,0.934517,0.934343,0.933263,0.85492


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import numpy as np
import pandas as pd

# Encode target labels

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Models

models = {

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

# Train and Evaluate

results = []

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    acc_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    mcc_scores = []

    for train_idx, test_idx in skf.split(
        X_nutrients,
        y_encoded
    ):

        # Split labels

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        # Nutrient Features

        X_train_nutrient = X_nutrients[train_idx, :]
        X_test_nutrient = X_nutrients[test_idx, :]

        scaler = StandardScaler()

        X_train_nutrient = scaler.fit_transform(
            X_train_nutrient
        )

        X_test_nutrient = scaler.transform(
            X_test_nutrient
        )

        # gpt2 Embeddings

        X_train_embed = embeddings[train_idx]
        X_test_embed = embeddings[test_idx]

        # Combine Features

        X_train = np.hstack(
            (
                X_train_nutrient,
                X_train_embed
            )
        )

        X_test = np.hstack(
            (
                X_test_nutrient,
                X_test_embed
            )
        )

        # SMOTE

        smote = SMOTE(
            random_state=42
        )

        X_train_resampled, y_train_resampled = (
            smote.fit_resample(
                X_train,
                y_train
            )
        )

        # Train

        model.fit(
            X_train_resampled,
            y_train_resampled
        )

        # Predict

        y_pred = model.predict(
            X_test
        )

        # Metrics

        acc_scores.append(
            accuracy_score(
                y_test,
                y_pred
            )
        )

        precision_scores.append(
            precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        f1_scores.append(
            f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        mcc_scores.append(
            matthews_corrcoef(
                y_test,
                y_pred
            )
        )

    results.append({

        "Model": model_name,

        "Accuracy":
            np.mean(acc_scores),

        "Precision":
            np.mean(precision_scores),

        "Recall":
            np.mean(recall_scores),

        "F1":
            np.mean(f1_scores),

        "MCC":
            np.mean(mcc_scores)
    })

# Results Table

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

print("\nFINAL RESULTS\n")

display(results_df)


Training Extra Trees...

FINAL RESULTS



,Model,Accuracy,Precision,Recall,F1,MCC
0,Extra Trees,0.914815,0.913867,0.914815,0.912945,0.80934
